# 🔗 Notebook 3 — Complete RAG Pipeline Demo

**DocuMind AI Portfolio Project**

End-to-end walkthrough of the RAG pipeline:
1. Initialise the full pipeline
2. Ingest sample documents
3. Query with source citations
4. Query expansion demo
5. MMR vs Similarity comparison
6. Contextual compression demo
7. Conversation memory demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Environment loaded')

## 1. Initialise the RAG Pipeline

In [ ]:
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(config_path='config/config.yaml')
print('Pipeline initialised!')
print(f'LLM: {pipeline.llm_chain_manager.provider}/{pipeline.llm_chain_manager.model_name}')
print(f'Embeddings: {pipeline.embedding_engine.provider}/{pipeline.embedding_engine.model_name}')

## 2. Ingest Sample Documents

In [ ]:
import json

report = pipeline.ingest_documents('data/raw/sample_docs')
print('Ingestion Report:')
print(json.dumps(report, indent=2))

## 3. Basic Q&A with Source Citations

In [ ]:
questions = [
    'What is the annual leave policy at TechCorp?',
    'What was TechCorp revenue in FY 2024?',
    'How do I install DocuMind AI?',
]

for q in questions:
    print(f'\n{'='*60}')
    print(f'Q: {q}')
    result = pipeline.query(q)
    print(f'A: {result["answer"]}')
    print(f'Sources: {[s["filename"] for s in result["sources"]]}')
    print(f'Confidence: {result["confidence_score"]:.0%} | Time: {result["retrieval_time_ms"]:.0f}ms retrieval')

## 4. Query Expansion (Multi-Query Retrieval)

In [ ]:
# Multi-query generates variations to improve recall
original_query = 'What are the performance review criteria?'

chain = pipeline.llm_chain_manager.build_query_expansion_chain()
try:
    result = chain.invoke({'question': original_query})
    variations = str(result.get('text', result)).strip().split('\n')
    print(f'Original query: {original_query}')
    print('\nGenerated variations:')
    for i, v in enumerate(variations[:4], 1):
        if v.strip():
            print(f'  {i}. {v.strip()}')
except Exception as e:
    print(f'Note: LLM not available for query expansion. Error: {e}')
    print('\nFallback: using original query only.')

## 5. MMR vs Similarity Search Comparison

In [ ]:
query = 'employee benefits and compensation'

sim_docs = pipeline.vector_store_manager.similarity_search(query, k=5)
mmr_docs = pipeline.vector_store_manager.mmr_search(query, k=5)

print('--- Similarity Search (pure relevance) ---')
sim_sources = set()
for d in sim_docs:
    fname = d.metadata.get('filename', '')
    sim_sources.add(fname)
    print(f'  [{fname}] {d.page_content[:80]}...')

print(f'\nUnique sources: {len(sim_sources)}')

print('\n--- MMR Search (relevance + diversity) ---')
mmr_sources = set()
for d in mmr_docs:
    fname = d.metadata.get('filename', '')
    mmr_sources.add(fname)
    print(f'  [{fname}] {d.page_content[:80]}...')

print(f'\nUnique sources: {len(mmr_sources)}')
print(f'\n💡 MMR retrieves from {len(mmr_sources)} unique sources vs {len(sim_sources)} for pure similarity.')

## 6. Conversation Memory Demo

In [ ]:
session_id = 'demo_session_001'

conversation = [
    'What is the annual leave policy?',
    'How many days can be carried forward to next year?',  # follow-up
    'What about sick leave?',  # another follow-up
]

for turn, q in enumerate(conversation, 1):
    print(f'\n[Turn {turn}] User: {q}')
    result = pipeline.query(q, session_id=session_id)
    print(f'[Turn {turn}] AI: {result["answer"][:200]}...')

print('\n--- Conversation History ---')
history = pipeline.conversation_manager.get_history(session_id)
for msg in history:
    role = '👤 User' if msg['role'] == 'human' else '🤖 AI'
    print(f'{role}: {msg["content"][:100]}...')

## 7. Document Summary

In [ ]:
summary = pipeline.get_document_summary('financial_report.pdf')
print('Financial Report Summary:')
print(summary)

## 8. Pipeline Health Check & Stats

In [ ]:
health = pipeline.health_check()
print('Health Check:')
for k, v in health['components'].items():
    status = '✅' if v == 'ok' else '⚠️'
    print(f'  {status} {k}: {v}')

stats = pipeline.get_pipeline_stats()
print(f'\nVector store: {stats["vector_store"]["total_vectors"]} vectors')
print(f'Active sessions: {stats["active_sessions"]}')
print(f'Ingested docs: {stats["ingested_docs"]}')